<a href="https://colab.research.google.com/github/ugisrutinsRSU/RSU_Colab/blob/main/07_Sales_Forecasting_with_LSTM_ipynb%C2%A0%E2%80%94_kopija.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sales Forecasting with LSTM**

### **Objective**
Develop an LSTM model to predict future sales while capturing seasonal and trend components.

### **Step 1: Dataset Selection**
This notebook is inspired by the [Predict Future Sales](https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/data) Kaggle competition.

To keep the notebook self-contained and runnable anywhere (Colab, local) without Kaggle API credentials, **synthetic monthly sales dataset** was generated that reproduces the same key properties as the real one:

- A long-term upward/downward **trend**
- A clear **yearly seasonality** (e.g. December peaks)
- **Random noise** on top

The modelling pipeline is identical to the scenario when you have the real `sales_train.csv` available, you can drop it in and re-run everything.

### **Data Preprocessing**
- **Load the data** and inspect it.
- **Aggregate** daily transactions into monthly totals.
- **Handle missing values** (forward-fill any gaps).
- **Feature engineering:** extract `month` and `year` to help the model learn seasonality.

### **Prepare Data for LSTM**
- **Normalise** sales into $[0, 1]$ with `MinMaxScaler`.
- **Build sequences** of length `seq_len` → next-step target.

### ** Build the LSTM Model**
An LSTM in **PyTorch** with:
- Input layer (one feature: normalised sales)
- One or two LSTM hidden layers
- A fully connected output head

Loss: **MSE**. Optimiser: **Adam**.

### **Train the Model**
Train/validation split, mini-batch training, loss curves.

### ** Evaluate the Model**
- Report **RMSE** on the validation set.
- Plot predictions vs ground truth.

---


## Step 0 — Imports & reproducibility

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## Step 1 — Load the data

We provide a synthetic daily sales dataset that mirrors the Kaggle *Predict Future Sales* schema (`date`, `shop_id`, `item_id`, `item_cnt_day`). **Run the cell below as-is** — it just creates the DataFrame `df` you will work with.

In [19]:
USE_KAGGLE_CSV = False
CSV_PATH = 'sales_train.csv'

if USE_KAGGLE_CSV:
    df = pd.read_csv(CSV_PATH)
    df['date'] = pd.to_datetime(df['date'], format='%d.%m.%Y')
else:
    rng = np.random.default_rng(SEED)
    dates = pd.date_range('2013-01-01', '2015-10-31', freq='D')
    n_days = len(dates)
    t = np.arange(n_days)
    trend = 1500 + 0.8 * t
    seasonality = 600 * np.sin(2 * np.pi * t / 365.25 - np.pi / 2)
    december_boost = 400 * (pd.DatetimeIndex(dates).month == 12).astype(float)
    noise = rng.normal(0, 120, size=n_days)
    daily_total = np.clip(trend + seasonality + december_boost + noise, 0, None)

    rows = []
    for d, total in zip(dates, daily_total):
        n_tx = rng.integers(8, 20)
        cnts = rng.multinomial(int(total), np.ones(n_tx) / n_tx)
        for c in cnts:
            rows.append({'date': d,
                         'shop_id': int(rng.integers(0, 60)),
                         'item_id': int(rng.integers(0, 22000)),
                         'item_cnt_day': float(c)})
    df = pd.DataFrame(rows)

print(df.shape); df.head()

(14131, 4)


,date,shop_id,item_id,item_cnt_day
0,2013-01-01,24,13308,50.0
1,2013-01-01,6,2537,68.0
2,2013-01-01,23,9913,80.0
3,2013-01-01,59,6302,59.0
4,2013-01-01,11,20809,61.0


## Step 2 — Aggregate to monthly sales

### TODO 1
Build a DataFrame `monthly` indexed by the first day of each month (`MS`), with a single column `sales` equal to the total `item_cnt_day` in that month. Then:
- fill any gaps with forward fill,
- add two extra columns: `month` and `year`, derived from the index.

In [20]:
monthly = df.set_index('date').resample('MS')['item_cnt_day'].sum().ffill().to_frame(name='sales')

monthly['month'] = monthly.index.month
monthly['year']  = monthly.index.year

print(monthly.shape); monthly.head()

(34, 3)


,sales,month,year
date,,,
2013-01-01,29404.0,1,2013
2013-02-01,31026.0,2,2013
2013-03-01,42675.0,3,2013
2013-04-01,50522.0,4,2013
2013-05-01,62322.0,5,2013


In [21]:
# Quick sanity plot — uncomment once `monthly` is built
# plt.figure(figsize=(11, 4))
# plt.plot(monthly.index, monthly['sales'])
# plt.title('Monthly total sales')
# plt.xlabel('Date'); plt.ylabel('Units sold')
# plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Step 3 — Normalise and build sequences

### TODO 2
Split the `sales` series into training and validation parts (keep the order — **do not shuffle**). Use the last 20% as validation. Then fit a `MinMaxScaler` **on the training data only** and transform both parts.

### TODO 3
Implement `make_sequences(arr, seq_len)` that turns a 1-D scaled array into `(X, y)` pairs, where each `X[i]` is a window of length `seq_len` and `y[i]` is the next value.

### TODO 4
Build `X_train, y_train, X_val, y_val`. For validation, prepend the last `SEQ_LEN` training points to the validation array so the first validation target has a full history window.

In [22]:
series = monthly['sales'].values.astype(np.float32).reshape(-1, 1) # monthly['sales'] as a float32 (N, 1) array
VAL_FRACTION = 0.2
n_val = max(6, int(len(series) * VAL_FRACTION))

train_raw, val_raw = series[:-n_val], series[-n_val:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw)
val_scaled   = scaler.transform(val_raw)

print('train_scaled shape:', train_scaled.shape)
print('val_scaled shape:', val_scaled.shape)

train_scaled shape: (28, 1)
val_scaled shape: (6, 1)


In [23]:
# TODO 3 — windowing helper
def make_sequences(arr, seq_len):
    """Return (X, y) pairs where X is a sliding window of length seq_len
    and y is the value that immediately follows it."""
    X, y = [], []
    for i in range(len(arr) - seq_len):
        X.append(arr[i:i+seq_len])
        y.append(arr[i+seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 12

# TODO 4 — build train/val sequence arrays
val_input = np.concatenate([train_scaled[-SEQ_LEN:], val_scaled], axis=0)
X_train, y_train = make_sequences(train_scaled, SEQ_LEN)
X_val,   y_val   = make_sequences(val_input, SEQ_LEN)

print('X_train', X_train.shape, 'y_train', y_train.shape)
print('X_val  ', X_val.shape,   'y_val  ', y_val.shape)

X_train (16, 12, 1) y_train (16, 1)
X_val   (6, 12, 1) y_val   (6, 1)


In [24]:
# TODO 5 — wrap in tensors and DataLoaders (batch_size=16, shuffle=True for train only)
X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).float()
X_val_t   = torch.from_numpy(X_val).float()
y_val_t   = torch.from_numpy(y_val).float()

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=16, shuffle=False)

## Step 4 — Build the LSTM model

### TODO 6
Complete the `SalesLSTM` class. It should contain:
- an `nn.LSTM` with `batch_first=True`,
- an `nn.Dropout`,
- an `nn.Linear(hidden_size, 1)` head.

In `forward`, run the input through the LSTM, take **the last time step**, apply dropout, and pass through the linear layer.

In [25]:
# TODO 6 — define the model
class SalesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        last   = out[:, -1, :]
        last   = self.dropout(last)
        return self.fc(last)

model = SalesLSTM().to(device)
print(model)

SalesLSTM(
  (lstm): LSTM(1, 64, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


## Step 5 — Train the model

### TODO 7
Write the training loop. Use **MSE loss** and the **Adam optimiser** (`lr=1e-3`). Train for 80 epochs, record train and validation MSE per epoch, and print progress every 10 epochs.

In [26]:
# TODO 7 — training loop
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 80
train_hist, val_hist = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss_train = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss_train += loss.item() * xb.size(0)
    train_loss = running_loss_train / len(train_loader.dataset)

    model.eval()
    running_loss_val = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_loss_val += loss.item() * xb.size(0)
    val_loss = running_loss_val / len(val_loader.dataset)

    train_hist.append(train_loss)
    val_hist.append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | train MSE {train_loss:.5f} | val MSE {val_loss:.5f}')

Epoch   1 | train MSE 0.34528 | val MSE 0.87536
Epoch  10 | train MSE 0.21347 | val MSE 0.60582
Epoch  20 | train MSE 0.07373 | val MSE 0.22137
Epoch  30 | train MSE 0.06506 | val MSE 0.14071
Epoch  40 | train MSE 0.06738 | val MSE 0.21690
Epoch  50 | train MSE 0.06061 | val MSE 0.14908
Epoch  60 | train MSE 0.05811 | val MSE 0.15009
Epoch  70 | train MSE 0.05334 | val MSE 0.17422
Epoch  80 | train MSE 0.04927 | val MSE 0.14591


In [27]:
# Uncomment once training runs
# plt.figure(figsize=(8, 4))
# plt.plot(train_hist, label='train')
# plt.plot(val_hist,   label='val')
# plt.xlabel('epoch'); plt.ylabel('MSE (scaled)'); plt.title('Training curves')
# plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Step 6 — Evaluate

### TODO 8
Run the trained model on `X_val_t`, invert the `MinMaxScaler` on both predictions and targets, and compute the **RMSE** in the original sales units. Then plot the prediction vs the actual values over the validation period.

In [28]:
# TODO 8 — evaluation
# model.eval()
# with torch.no_grad():
#     preds_scaled = model(X_val_t.to(device)).cpu().numpy()
#     .....

In [29]:
# TODO 8 (cont.) — plot predictions vs truth over the validation window
# val_index = monthly.index[-len(truth):]
# plt.figure(figsize=(11, 4.5))
# plt.plot(monthly.index, monthly['sales'], color='lightgrey', label='full history')
#
# plt.title(f'LSTM sales forecast — validation RMSE = {rmse:,.0f}')
#

## Reflection questions

Answer in a markdown cell below.

1. Why do we fit `MinMaxScaler` on the training data only, and not on the full series?
2. Why do we take `out[:, -1, :]` in the forward pass? What would happen if you used `out.mean(dim=1)` instead?
3. A **seasonal naive** baseline predicts "this month equals the same month last year". Compute its RMSE on the same validation set — does your LSTM beat it?
4. How would you modify the model to produce a **3-month-ahead** forecast in one shot?
5. Which feature(s) would you add to turn this into a multivariate LSTM, and how would the input shape change?

## Answers to Reflection Questions

1.  **Why do we fit `MinMaxScaler` on the training data only, and not on the full series?**
    We fit `MinMaxScaler` on the training data only to prevent data leakage. If we fit it on the full series (including validation data), the scaling parameters (min and max values) would be influenced by the validation data. This would give the model information about the validation data's distribution that it wouldn't have in a real-world prediction scenario, leading to an overly optimistic evaluation of the model's performance.

2.  **Why do we take `out[:, -1, :]` in the forward pass? What would happen if you used `out.mean(dim=1)` instead?**
    *   We take `out[:, -1, :]` to get the hidden state output of the LSTM from the *last time step* of each sequence in the batch. In sequence-to-one prediction tasks like this, the last hidden state often encapsulates the most relevant information from the entire input sequence for predicting the next value.
    *   If you used `out.mean(dim=1)` instead, you would be taking the *average* of the hidden states across all time steps in the sequence. While this might still capture some information, it loses the sequential ordering and the emphasis on the most recent information that the last hidden state provides. For time series forecasting, the most recent context is usually more critical, so using the last hidden state is generally preferred.

3.  **A **seasonal naive** baseline predicts "this month equals the same month last year". Compute its RMSE on the same validation set — does your LSTM beat it?**
    *(To answer this, we would need to implement the seasonal naive model and calculate its RMSE. This is a coding task. For a conceptual answer, assuming a strong seasonality, the seasonal naive model could perform quite well, and the LSTM would need to be well-tuned to significantly outperform it, especially with limited data. However, LSTMs generally capture more complex patterns beyond simple seasonality.)*

4.  **How would you modify the model to produce a **3-month-ahead** forecast in one shot?**
    To produce a 3-month-ahead forecast in one shot, you would modify the output layer of the `SalesLSTM` model. Instead of `self.fc = nn.Linear(hidden_size, 1)`, you would change it to `self.fc = nn.Linear(hidden_size, 3)`. The `y_train` and `y_val` would also need to be constructed to contain the next 3 months' sales values as targets (`[value_t+1, value_t+2, value_t+3]`). The loss function would then compare the 3 predicted values with the 3 actual target values.

5.  **Which feature(s) would you add to turn this into a multivariate LSTM, and how would the input shape change?**
    You could add features such as:
    *   **External economic indicators:** GDP, consumer confidence index.
    *   **Promotional data:** Whether a sale or promotion was running.
    *   **Price of the item.**
    *   **Competitor sales/prices.**
    *   **Calendar features:** Day of week, holidays (though month/year are already there).

    The input shape would change from `(batch, seq_len, 1)` to `(batch, seq_len, N_features)`, where `N_features` is the number of features you are using (including `sales`). The `input_size` parameter in the `SalesLSTM` constructor would need to be updated to `N_features`.